### Day 17 · EDA 综合项目

**EDA**(Exploratory Data Analysis)是拿到数据后的第一步。本节用一份硬编码的员工绩效数据集,完整走完 EDA 全流程: 数据概览 -> 缺失值处理 -> 异常值检测 -> 分组统计 -> 可视化。每个 cell 独立 import、独立运行、数据硬编码。

#### 数据概览 head / info / describe

[痛点] 拿到一份新数据集,不知道从哪下手 -- 不知道有多少行、哪些列有缺失、数值分布如何。

[类比] 数据概览像体检报告: `shape` 是身高体重(整体规模)、`info()` 是血常规(各列类型与缺失)、`describe()` 是各项指标参考值(均值、分位数)。

[解释] `head(n)` 看前 n 行原始数据; `info()` 显示列名、非空数量、数据类型; `describe()` 对数值列给出计数、均值、标准差、最小值、四分位、最大值。

In [ ]:
import pandas as pd
import numpy as np

# --- 执行过程 ---
# ① 构造硬编码数据字典,包含 8 条员工记录
data = {
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十"],
    "部门": ["销售", "技术", "技术", "市场", "财务", "销售", "技术", "市场"],
    "薪资": [8000, 15000, 13000, 9000, 11000, 7500, 16000, 8500],
    "年龄": [25, 32, 29, 30, 35, 28, 31, 27],
    "绩效评分": [78, 92, 88, 65, 74, 82, 95, 70]
}
# ② pd.DataFrame(data): 将字典转为 DataFrame,键为列名
df = pd.DataFrame(data)

# ③ head(3): 查看前 3 行,快速了解数据结构
print("--- head(3) ---")
print(df.head(3))
print("")
# ④ info(): 显示列名、非空数量、数据类型、内存占用
print("--- info() ---")
df.info()
print("")
# ⑤ describe(): 对数值列给出均值、标准差、四分位等
print("--- describe() ---")
print(df.describe())

In [ ]:
import pandas as pd
import numpy as np

# --- 执行过程 ---
# ① 构造含 np.nan 的缺失值数据
data = {
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "年龄": [25, np.nan, 29, np.nan, 35],
    "薪资": [8000, 15000, np.nan, 9000, 11000]
}
df = pd.DataFrame(data)

# ② isnull(): 返回布尔 DataFrame,缺失位置为 True
print("--- isnull() ---")
print(df.isnull())
print("")
# ③ isnull().sum(): 统计每列缺失值数量
print("--- isnull().sum() ---")
print(df.isnull().sum())

# ④ 计算缺失比例 = 缺失数 / 总行数 * 100
missing_ratio = (df.isnull().sum() / len(df) * 100).round(1)
print("")
print("--- 缺失比例(%) ---")
print(missing_ratio)

**逐行解剖**

- `head(n)`: 默认 n=5,用于快速预览数据,避免打印几千行。
- `info()`: 最常用于检查缺失值(看 Non-Null Count)和类型(看 Dtype)。
- `describe()`: 默认只统计数值列,给出 mean/std/min/25%/50%/75%/max。
- `isnull().sum()`: 统计每列缺失值数量,是数据清洗的第一步。

**练习 1**

已有以下 DataFrame,请完成: 1. 查看前 3 行; 2. 统计每列缺失值数量; 3. 计算数值列的描述统计。

> 问自己:
> - `head()` 默认显示几行?
> - `describe()` 默认统计哪些列?

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [25, np.nan, 29, 30],
    "薪资": [8000, 15000, np.nan, 9000]
})

# ======================
# 学员代码区
# ======================
pass

# ======================
# 参考答案
# ======================
print("--- 前 3 行 ---")
print(df.head(3))
print("--- 缺失值统计 ---")
print(df.isnull().sum())
print("--- 描述统计 ---")
print(df.describe())

#### 缺失值处理 fillna / dropna

[痛点] 数据中有缺失值(NaN),直接计算会报错或得到错误结果。需要决定是删除还是填充。

[类比] 缺失值像考卷上的空白题: `dropna` 是整页撕掉(删除行); `fillna` 是猜一个答案填上去(填充)。有异常值时用 median 填,没有时用 mean 填;分类变量用 mode(众数)填。

[解释] `fillna(value)` 用指定值填充缺失,常用 median() 或 mean(); `dropna()` 删除含缺失值的行或列,`how="any"` 表示只要有一个 NaN 就删,`how="all"` 表示全为 NaN 才删。

In [ ]:
import pandas as pd
import numpy as np

# --- 执行过程 ---
# ① 构造含缺失值的数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "年龄": [25, np.nan, 29, np.nan, 35],
    "薪资": [8000, 15000, np.nan, 9000, 11000]
})

# ② median(): 计算中位数,对异常值更鲁棒
age_median = df["年龄"].median()
print("年龄中位数:", age_median)
# ③ fillna(median): 用中位数填充年龄缺失值
df["年龄"] = df["年龄"].fillna(age_median)
print("--- 年龄填充后 ---")
print(df)

# ④ mean(): 计算薪资均值
salary_mean = df["薪资"].mean()
print("薪资均值:", salary_mean)
# ⑤ fillna(mean): 用均值填充薪资缺失值
df["薪资"] = df["薪资"].fillna(salary_mean)
print("--- 薪资填充后 ---")
print(df)

In [ ]:
import pandas as pd
import numpy as np

# --- 执行过程 ---
# ① 构造含缺失值的示例数据
df = pd.DataFrame({
    "A": [1, np.nan, 3],
    "B": [4, 5, np.nan],
    "C": [7, 8, 9]
})

# ② dropna(): 默认 axis=0,删除含 NaN 的行
print("--- dropna() 删除行 ---")
print(df.dropna())
print("")
# ③ dropna(axis=1): 删除含 NaN 的列
print("--- dropna(axis=1) 删除列 ---")
print(df.dropna(axis=1))

# ④ how="all": 仅当整行都是 NaN 时才删除
df2 = pd.DataFrame({
    "A": [1, np.nan, np.nan],
    "B": [4, np.nan, 5]
})
print("")
print("--- how=all ---")
print(df2.dropna(how="all"))

**逐行解剖**

- `fillna(value)`: 用 value 填充所有 NaN,返回新 DataFrame(不修改原数据)。
- `median()`: 中位数,不受极端值影响,适合有异常值的列。
- `mean()`: 均值,适合没有异常值的列。
- `dropna(axis=0)`: 删除含 NaN 的行; `axis=1` 删除含 NaN 的列。
- `how="any"`: 只要有一个 NaN 就删(默认); `how="all"`: 全为 NaN 才删。

**练习 2**

已有以下 DataFrame,请完成: 1. 用中位数填充"年龄"列的缺失值; 2. 用均值填充"薪资"列的缺失值; 3. 删除"城市"列中仍为缺失值的行(假设城市列有缺失)。

> 问自己:
> - `fillna` 会修改原 DataFrame 吗?
> - 如果想直接修改原数据,应该怎么做?

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "年龄": [25, np.nan, 29, 30],
    "薪资": [8000, 15000, np.nan, 9000],
    "城市": ["北京", "上海", np.nan, "广州"]
})

# ======================
# 学员代码区
# ======================
pass

# ======================
# 参考答案
# ======================
df["年龄"] = df["年龄"].fillna(df["年龄"].median())
df["薪资"] = df["薪资"].fillna(df["薪资"].mean())
df = df.dropna(subset=["城市"])
print(df)

#### 异常值检测 3 sigma 原则

[痛点] 数据中存在极端值(如薪资 120000 混入普通员工数据),会严重拉高均值,导致分析结论失真。

[类比] 异常值像考试中的满分或零分 -- 如果全班都在 60-90 分,突然出现一个 5 分,很可能是填错了答题卡,需要检查。

[解释] 3 sigma 原则: 对于近似正态分布的数据,约 99.7% 的值落在 [均值-3*标准差, 均值+3*标准差] 区间内。超出此范围的值视为异常值。

In [ ]:
import pandas as pd
import numpy as np

# --- 执行过程 ---
# ① 构造含极端值(120000)的薪资数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十"],
    "薪资": [8000, 15000, 13000, 9000, 11000, 7500, 16000, 120000]
})

# ② mean() 和 std(): 计算均值和标准差
mu = df["薪资"].mean()
sigma = df["薪资"].std()
print("均值:", mu)
print("标准差:", sigma)

# ③ 计算 3 sigma 上下界
lower = mu - 3 * sigma
upper = mu + 3 * sigma
print("3 sigma 范围:", lower, ",", upper)

# ④ 筛选异常值: 超出 3 sigma 范围的行
outliers = df[(df["薪资"] < lower) | (df["薪资"] > upper)]
print("--- 异常值 ---")
print(outliers)

# ⑤ 筛选正常值: 在 3 sigma 范围内的行
normal = df[(df["薪资"] >= lower) & (df["薪资"] <= upper)]
print("--- 正常值 ---")
print(normal)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 执行过程 ---
# ① 设置中文字体,解决中文显示为方框的问题
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

df = pd.DataFrame({
    "薪资": [8000, 15000, 13000, 9000, 11000, 7500, 16000, 120000]
})

# ② boxplot(): 用箱线图可视化异常值,离群点显示为单独的点
df[["薪资"]].boxplot()
plt.title("薪资箱线图(异常值检测)")
plt.ylabel("薪资(元)")
plt.show()

# ③ hist(): 直方图展示薪资分布形态
df["薪资"].hist(bins=8, edgecolor="black")
plt.title("薪资分布直方图")
plt.xlabel("薪资(元)")
plt.ylabel("频数")
plt.show()

**逐行解剖**

- `mean()`: 均值,受异常值影响大。
- `std()`: 标准差,衡量数据波动程度。
- `mu - 3*sigma` / `mu + 3*sigma`: 3 sigma 区间的下界和上界。
- `df[(df["列"] < lower) | (df["列"] > upper)]`: 筛选异常值,`|` 表示或。
- `boxplot()`: 箱线图,超出 1.5 倍 IQR 的值显示为离群点。

**练习 3**

已有以下 DataFrame,请完成: 1. 用 3 sigma 原则检测"薪资"列的异常值; 2. 打印出异常值的姓名和薪资; 3. 计算去除异常值后的平均薪资。

> 问自己:
> - 3 sigma 区间怎么算?
> - 如何筛选出正常值(非异常值)?

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七"],
    "薪资": [8000, 15000, 13000, 9000, 120000]
})

# ======================
# 学员代码区
# ======================
pass

# ======================
# 参考答案
# ======================
mu = df["薪资"].mean()
sigma = df["薪资"].std()
lower = mu - 3 * sigma
upper = mu + 3 * sigma

outliers = df[(df["薪资"] < lower) | (df["薪资"] > upper)]
print("异常值:")
print(outliers[["姓名", "薪资"]])

normal = df[(df["薪资"] >= lower) & (df["薪资"] <= upper)]
print("去除异常值后平均薪资:", normal["薪资"].mean())

#### 分组统计 groupby

[痛点] 想知道不同部门的平均薪资、各部门人数,只能手动筛选再计算,效率极低。

[类比] groupby 像把一筐水果按种类分堆: 先按"颜色"分组,再数每组有几个、平均重量多少。

[解释] `groupby("列")` 按某列分组,然后对每组做聚合运算(mean/sum/count/max/min)。`agg()` 可以一次性做多种聚合,支持命名聚合自定义输出列名。

In [ ]:
import pandas as pd

# --- 执行过程 ---
# ① 构造包含部门、薪资、绩效的数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十"],
    "部门": ["销售", "技术", "技术", "市场", "财务", "销售", "技术", "市场"],
    "薪资": [8000, 15000, 13000, 9000, 11000, 7500, 16000, 8500],
    "绩效评分": [78, 92, 88, 65, 74, 82, 95, 70]
})

# ② groupby("部门"): 按部门分组
# ③ ["薪资"].mean(): 计算每组的平均薪资
dept_salary = df.groupby("部门")["薪资"].mean()
print("--- 各部门平均薪资 ---")
print(dept_salary.round(0))

# ④ count(): 统计每组人数
dept_count = df.groupby("部门")["姓名"].count()
print("--- 各部门人数 ---")
print(dept_count)

In [ ]:
import pandas as pd

# --- 执行过程 ---
# ① 构造数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六", "孙七", "周八", "吴九", "郑十"],
    "部门": ["销售", "技术", "技术", "市场", "财务", "销售", "技术", "市场"],
    "薪资": [8000, 15000, 13000, 9000, 11000, 7500, 16000, 8500],
    "绩效评分": [78, 92, 88, 65, 74, 82, 95, 70]
})

# ② agg(): 一次性做多种聚合
# ③ 命名聚合: 新列名=("原列名", "函数名")
result = df.groupby("部门").agg(
    薪资均值=("薪资", "mean"),
    薪资最高=("薪资", "max"),
    绩效均值=("绩效评分", "mean"),
    人数=("姓名", "count")
)
print("--- 各部门多维度聚合 ---")
print(result.round(1))

**逐行解剖**

- `groupby("列")`: 按指定列分组,返回 GroupBy 对象,需要配合聚合函数使用。
- `df.groupby("A")["B"].mean()`: 按 A 分组后,对 B 列求均值。
- `agg()`: 同时应用多个聚合函数,传入列表或命名聚合。
- `命名聚合`: 格式为 `新列名=("原列名", "函数名")`,输出列名更清晰。

**练习 4**

已有以下 DataFrame,请完成: 1. 按"部门"分组,计算各部门的平均绩效评分; 2. 使用 agg() 同时计算各部门的薪资均值、最高薪资和人数。

> 问自己:
> - groupby 后如果不加聚合函数会怎样?
> - 命名聚合的格式是什么?

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "部门": ["销售", "技术", "技术", "销售"],
    "薪资": [8000, 15000, 13000, 9000],
    "绩效评分": [78, 92, 88, 65]
})

# ======================
# 学员代码区
# ======================
pass

# ======================
# 参考答案
# ======================
print("--- 各部门平均绩效 ---")
print(df.groupby("部门")["绩效评分"].mean())

print("--- 多维度聚合 ---")
result = df.groupby("部门").agg(
    薪资均值=("薪资", "mean"),
    薪资最高=("薪资", "max"),
    人数=("姓名", "count")
)
print(result)

#### 可视化 plot / bar / hist

[痛点] 数据清洗完了,但光看数字无法直观发现规律 -- 需要图表来展示分布和对比。

[类比] 可视化像给数据拍照: `hist` 是拍全景(看整体分布)、`bar` 是拍合影(对比不同类别)、`plot` 是拍视频(看趋势变化)。

[解释] `hist()` 画直方图展示数值分布; `bar()` 画柱状图对比不同类别; `plot()` 画折线图展示趋势。Pandas 的绘图方法基于 matplotlib,可直接调用。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- 执行过程 ---
# ① 设置中文字体
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

df = pd.DataFrame({
    "薪资": [8000, 15000, 13000, 9000, 11000, 7500, 16000, 8500]
})

# ② hist(): 直方图,展示薪资分布,bins=5 分成 5 个区间
df["薪资"].hist(bins=5, edgecolor="black", color="skyblue")
plt.title("薪资分布直方图")
plt.xlabel("薪资(元)")
plt.ylabel("员工数")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- 执行过程 ---
# ① 设置中文字体
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

df = pd.DataFrame({
    "部门": ["销售", "技术", "技术", "市场", "财务", "销售", "技术", "市场"],
    "薪资": [8000, 15000, 13000, 9000, 11000, 7500, 16000, 8500]
})

# ② groupby + mean: 计算各部门平均薪资
dept_avg = df.groupby("部门")["薪资"].mean()
# ③ plot(kind="bar"): 柱状图,每根柱子代表一个部门
dept_avg.plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
plt.title("各部门平均薪资")
plt.xlabel("部门")
plt.ylabel("平均薪资(元)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- 执行过程 ---
# ① 设置中文字体
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

df = pd.DataFrame({
    "月份": ["1月", "2月", "3月", "4月", "5月"],
    "销售额": [120, 150, 170, 160, 200]
})

# ② plot(): 默认折线图,marker="o" 在每个数据点画圆点
plt.plot(df["月份"], df["销售额"], marker="o", color="red")
plt.title("月度销售额趋势")
plt.xlabel("月份")
plt.ylabel("销售额(万元)")
plt.tight_layout()
plt.show()

**逐行解剖**

- `hist(bins=n)`: 直方图,bins 控制区间数量,edgecolor 给柱子描边。
- `plot(kind="bar")`: 柱状图,`color` 传列表给每根柱子指定颜色。
- `xticks(rotation=0)`: 让 x 轴标签水平显示,不旋转。
- `tight_layout()`: 自动调整布局,防止标签被截断。
- `plt.rcParams`: 设置中文字体,解决中文显示为方框的问题。

**练习 5**

已有以下 DataFrame,请完成: 1. 画出"薪资"列的直方图(bins=6); 2. 画出各部门平均薪资的柱状图。

> 问自己:
> - 直方图和柱状图有什么区别?
> - 柱状图之前需要先做什么操作?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "部门": ["销售", "技术", "技术", "销售"],
    "薪资": [8000, 15000, 13000, 9000]
})

# ======================
# 学员代码区
# ======================
pass

# ======================
# 参考答案
# ======================
df["薪资"].hist(bins=6, edgecolor="black")
plt.title("薪资分布")
plt.xlabel("薪资(元)")
plt.ylabel("人数")
plt.show()

dept_avg = df.groupby("部门")["薪资"].mean()
dept_avg.plot(kind="bar")
plt.title("各部门平均薪资")
plt.xlabel("部门")
plt.ylabel("平均薪资(元)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**今日小结**

| 概念 | 解决的痛点 |
|---|---|
| head / info / describe | 拿到新数据不知从哪下手 |
| fillna / dropna | 缺失值导致计算报错或失真 |
| 3 sigma 异常值检测 | 极端值拉高均值,结论失真 |
| groupby + agg | 手动分组计算效率低 |
| plot / bar / hist | 光看数字无法直观发现规律 |

**更多练习**

- 当堂练: course/lesson17/in_class/practice01-06.py
- 课后作业: course/lesson17/homework/task01-03.py